# Practice Set: Classification Models (Decision Tree, KNN)
**SkillNexis ML & AI Internship, Week 2**

Dataset: Titanic (`data/titanic_train.csv`, same file as Assignment 2).

| Question | Where |
|---|---|
| Q1. Train a logistic regression model on Titanic | `02_titanic_logistic_regression.ipynb` |
| Q2. Confusion matrix and accuracy score | `02_titanic_logistic_regression.ipynb` (also repeated below for all models) |
| Q3. Compare DecisionTreeClassifier and KNeighborsClassifier accuracy | Section 2 |
| Q4. Tune `max_depth` of the Decision Tree and note the difference | Section 3 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

RANDOM_STATE = 42
Path("images").mkdir(exist_ok=True)

## 1. Data (same preparation as Assignment 2)

In [ ]:
DATA_PATH = Path("data/titanic_train.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Download train.csv from Kaggle (Titanic) and save it as data/titanic_train.csv")

df = pd.read_csv(DATA_PATH)
numeric_cols = ["Age", "SibSp", "Parch", "Fare"]
categorical_cols = ["Pclass", "Sex", "Embarked"]

X = df[numeric_cols + categorical_cols].copy()
X["Pclass"] = X["Pclass"].astype(str)
y = df["Survived"]

def make_preprocess():
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), numeric_cols),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_cols),
    ])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

## 2. Q3: Decision Tree vs KNN
KNN works on distances, so features must be scaled (done in the preprocessing step above).
Trees are unaffected by scaling. Both use default settings here (`max_depth=None`, `n_neighbors=5`).

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
}

rows, fitted = [], {}
for name, clf in models.items():
    pipe = Pipeline([("prep", make_preprocess()), ("model", clf)]).fit(X_train, y_train)
    fitted[name] = pipe
    rows.append({
        "Model": name,
        "Train accuracy": accuracy_score(y_train, pipe.predict(X_train)),
        "Test accuracy": accuracy_score(y_test, pipe.predict(X_test)),
        "5-fold CV accuracy": cross_val_score(
            Pipeline([("prep", make_preprocess()), ("model", clf)]), X_train, y_train, cv=5).mean(),
    })

comparison = pd.DataFrame(rows).set_index("Model").round(4)
comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, pipe) in zip(axes, fitted.items()):
    cm = confusion_matrix(y_test, pipe.predict(X_test))
    ConfusionMatrixDisplay(cm, display_labels=["Died", "Survived"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.savefig("images/practice_confusion_matrices.png", dpi=150)
plt.show()

**Observation (check against your output):** an unrestricted Decision Tree typically scores almost 100% on the
training data but noticeably lower on the test data. That gap is overfitting. Compare the train and test columns above.

## 3. Q4: Tune `max_depth` of the Decision Tree
A depth that is too small underfits, and one that is too large overfits. We pick the depth with **cross-validation on
the training set**, so the test set stays untouched until the final check.

In [ ]:
depths = range(1, 21)
train_acc, cv_acc, test_acc = [], [], []

for d in depths:
    pipe = Pipeline([("prep", make_preprocess()),
                     ("model", DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE))])
    pipe.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, pipe.predict(X_train)))
    test_acc.append(accuracy_score(y_test, pipe.predict(X_test)))
    cv_acc.append(cross_val_score(pipe, X_train, y_train, cv=5).mean())

tuning = pd.DataFrame({"max_depth": list(depths), "train": train_acc, "cv": cv_acc, "test": test_acc}).round(4)
tuning

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(tuning["max_depth"], tuning["train"], "o-", label="Train")
plt.plot(tuning["max_depth"], tuning["cv"], "s-", label="Cross-validation")
plt.plot(tuning["max_depth"], tuning["test"], "^--", label="Test", alpha=0.6)
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Decision Tree: effect of max_depth")
plt.xticks(list(depths))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("images/decision_tree_max_depth.png", dpi=150)
plt.show()

In [ ]:
best_depth = int(tuning.loc[tuning["cv"].idxmax(), "max_depth"])

default_tree = fitted["Decision Tree"]
tuned_tree = Pipeline([("prep", make_preprocess()),
                       ("model", DecisionTreeClassifier(max_depth=best_depth, random_state=RANDOM_STATE))]).fit(X_train, y_train)

final = pd.DataFrame({
    "max_depth": ["None (default)", best_depth],
    "Train accuracy": [accuracy_score(y_train, default_tree.predict(X_train)),
                       accuracy_score(y_train, tuned_tree.predict(X_train))],
    "Test accuracy": [accuracy_score(y_test, default_tree.predict(X_test)),
                      accuracy_score(y_test, tuned_tree.predict(X_test))],
}, index=["Default tree", "Tuned tree"]).round(4)

print(f"Best max_depth by cross-validation: {best_depth}")
final

## 4. Conclusions
Fill these in from *your* output:
* Decision Tree vs KNN test accuracy: **...** vs **...**
* Best `max_depth`: **...**; test accuracy changed from **...** to **...**
* The train-test gap shrank from **...** to **...**, which shows that limiting depth reduces overfitting.